In [1]:
import numpy as np
import pandas as pd

In [2]:
SEED = 42
rng = np.random.default_rng(SEED)

In [3]:
STAGIONI = ["PE23", "AI23", "PE24", "AI24", "PE25", "AI25"]

In [4]:
PREZZI = {
    "t-shirt": (35 ,60), "camicia": (70, 140), "maglia": (80, 180),
    "felpa": (70, 130), "pantalone": (90, 170), "jeans": (100, 180), 
    "giacca": (200, 450), "capospalla": (250, 600), "abito": (350, 500),
    "scarpe eleganti": (180, 350), "scarpe casual": (120, 150), 
    "cravatta": (50, 100), "calzini": (30, 40), "profumo": (200, 280),
    "costume": (40, 90),
}

In [5]:
SOLO_AI = {"capospalla"}
SOLO_PE = {"costume"}

In [6]:
COLORI = ["nero", "blu", "grigio", "bianco", "beige", "marrone", "verde", "rosso", "fantasia"]

In [13]:
BRAND = pd.DataFrame([
    ("Ago&Filo", "Ago&Filo S.r.l.", ["camicia", "cravatta"], 2.6, "regolare", 0.003, 0.75, 1.00, "fermo"), 
    ("Lardoni", "Lardoni Sartoria S.p.a.", ["giacca", "pantalone", "abito"], 2.4, "veste piccolo", 0.004, 0.60, 1.00, "fermo"),
    ("Dipartimento 3", "Dipartimento Italia", ["felpa", "t-shirt", "jeans"], 2.5, "regolare", 0.013, 0.60, 1.00, "fermo"),
    ("FreePlace Outerwear", "FreePlace Distribuzione", ["capospalla", "giacca"], 2.3, "regolare", 0.003, 0.35, 1.00, "fermo"),
    ("Cruciverba", "Maglificio Cruciverba", ["maglia", "felpa"], 2, "veste piccolo", 0.003, 0.70, 1.00, "fermo"),
    ("Hole", "Hole Footwear", ["scarpe casual"], 2.2, "regolare", 0.005, 0.55, 0.80, "conto_vendita"), 
    ("Borris Calzature", "Calzaturificio Borris", ["scarpe eleganti"], 2.1, "regolare", 0.004, 0.55, 1.00, "fermo"),
    ("Lieve", "Lieve Profumi", ["profumo"], 2.0, "n.d.", 0.005, 0.62, 1.00, "fermo"),
    ("Grillo", "Grillo Accessori", ["calzini", "cravatta"], 2.8, "n.d.", 0.005, 0.70, 1.00, "fermo"),
    ("Lido Club", "Lido Club Distribuzione", ["t-shirt", "costume", "pantalone"], 2.5, "regolare", 0.007, 0.40, 1.00, "fermo"),
], columns=["brand", "fornitore", "categorie", "ricarico", "vestibilita", "tasso_difetti", "appeal", "fill_rate", "modalita"])

In [8]:
def prezzo_commerciale(x):
    """Arrotonda a un prezzo da cartellino: finale in 9 (es. 69, 109, 219)."""
    return float(np.ceil(x / 10) *10 -1) if x >= 20 else float(round(x) - 0.1)

In [9]:
prezzo_commerciale(116.34)

119.0

In [10]:
def crea_catalogo(articoli_per_categoria=(6,10)):
    righe = []
    for _, b in BRAND.iterrows():
        sigla = b["brand"][:3].upper()
        n = 0
        for stagione in STAGIONI:
            for cat in b["categorie"]:
                if (cat in SOLO_AI and stagione.startswith("PE")) or \
                   (cat in SOLO_PE and stagione.startswith("AI")):
                    continue
                for _ in range(rng.integers(*articoli_per_categoria, endpoint=True)):
                    n += 1
                    lo, hi = PREZZI[cat]
                    listino = prezzo_commerciale(rng.uniform(lo, hi))
                    costo = round(listino / (b["ricarico"] * rng.uniform(0.95, 1.05)), 2)
                    righe.append({
                        "articolo": f"{sigla}-{cat[:3].upper()}-{n:04d}",
                        "brand": b["brand"],
                        "fornitore": b["fornitore"],
                        "modalita_acquisto": b["modalita"],
                        "categoria": cat,
                        "colore": "n.d." if cat == "profumo" else rng.choice(COLORI),
                        "stagione": stagione,
                        "prezzo_listino": listino,
                        "costo_unitario": costo,
                    })
    return pd.DataFrame(righe)

In [11]:
cat = crea_catalogo()
print(cat.shape)
print(cat.head(8).to_string())
print(cat.groupby("brand").agg(articoli=("articolo", "count"),
                                    listino_medio=("prezzo_listino", "mean")).round(1))
print("Articoli duplicati:", cat["articolo"].duplicated().sum())
print("Costo >= listino:", (cat["costo_unitario"] >= cat["prezzo_listino"]).sum())
    

(909, 9)
       articolo     brand        fornitore modalita_acquisto categoria    colore stagione  prezzo_listino  costo_unitario
0  AGO-CAM-0001  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia     verde     PE23           109.0           40.47
1  AGO-CAM-0002  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia     beige     PE23           119.0           47.71
2  AGO-CAM-0003  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia  fantasia     PE23           129.0           48.24
3  AGO-CAM-0004  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia     beige     PE23            79.0           30.54
4  AGO-CAM-0005  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia    bianco     PE23           139.0           52.70
5  AGO-CAM-0006  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia     beige     PE23           129.0           49.90
6  AGO-CRA-0007  Ago&Filo  Ago&Filo S.r.l.             fermo  cravatta     rosso     PE23            79.0           31.77
7  AGO-CRA-0008

In [12]:
cat["modalita_acquisto"].value_counts()

modalita_acquisto
fermo            863
conto_vendita     46
Name: count, dtype: int64